# Generate the files

## Requirements

[rustup](https://rustup.rs/)
[python3](https://www.python.org/downloads/)

Use https://docs.astral.sh/uv/ and `uv sync` from the root of this project.

## Generate snapshot

The snapshot is partitioned by the day the games were played, `ipcs/year=YYYY/month=MM/day=DD/*.arrow`,
and generating it again only reads the replays that are not in it yet, use `--recreate` to start from
scratch. `snapshot.py` next to this notebook reads it, `s2.scan(ipc_dir, "details")`.

### Use swarmy to:
- Locate the SC2Replay directory of choice.
- Optimize it into a snapshot, written to `ipcs` in the replay directory.
- For the jupyter notebooks downloading the caches is not needed (yet).

### Use by hand/CLI

```bash
$ git clone https://github.com/sebosp/s2protocol-rs
$ cd s2protocol-rs
$ mkdir -p $LOCATION/ipcs/
$ cargo run --example full -r --no-default-features --features=dep_arrow,tracing_off -- \
    -v error --timing --source $HOME/SCReplaysOnNVMe/ --output $LOCATION/ipcs/ \
    --cache-path $HOME/SCReplaysOnNVMe/swarmy/ write-arrow-ipc --process-max-files 1000000
Located 59696 matching files by extension
0 replays are already in the snapshot, 6413 are copies of another replay, 17132 could not be read, 36151 new replays have valid init data, processing...
Wrote 36151 replays into 2090 days after 282.879022341s
Total time: 340.087666597s

$ ls ipcs/
metadata.json  year=2020  year=2021  year=2022  year=2023  year=2024  year=2025  year=2026
$ ls ipcs/year=2026/month=05/day=28/
cmd_target_point.arrow  cmd_target_unit.arrow  details.arrow  metadata.json  stats.arrow  unit_born.arrow  unit_died.arrow  upgrades.arrow

# somewhere else
$ git clone https://github.com/sebosp/s2-polars-data-analysis
$ cd s2-polars-data-analysis/
$ uv sync
$ uv run jupyter lab --notebook-dir=jupyter_notebooks/
# This should open jupyterlab.
# Running the first cell should open the duckdb app.
# Open the URL explained in the terminal, this will open this notebook for interacting with the data.
```

See s2protocol-rs `bench.md` for the time it takes on different machines.

In [ ]:
# Updated to the s2protocol-rs 3.6.0 snapshot, partitioned by day and keyed by ext_fs_id.
# import datashader as ds
import plotly.express as px
import polars as pl
import duckdb

# Reads the snapshot, see its docstrings.
import snapshot as s2

pl.Config.set_tbl_width_chars(256)
pl.Config.set_fmt_str_lengths(256)
pl.Config.set_tbl_rows(24)

duckdb.execute("CALL start_ui()")
# Location of the snapshot generated from above, `ipcs` in the replay directory with swarmy.
ipc_dir = "/home/seb/SC2Replays/PRO/RSL/ipcs"

details_df = s2.scan(ipc_dir, "details")

# Filter out these:
# Beacon*: Similiar to above, default targets of hatcheries and buildings
#          points to Beacons, clicks on the map, or actions when teams play
#          together and send attack/defend points on the map.
#          For now we'll avoid them.
unit_born_df = s2.scan(ipc_dir, "unit_born")
# .filter((~pl.col("unit_type_name").str.starts_with("Beacon")))
unit_died_df = s2.scan(ipc_dir, "unit_died")
# .filter((~pl.col("unit_died_name").str.starts_with("Beacon")))
stats_df = s2.scan(ipc_dir, "stats")
upgrades_df = s2.scan(ipc_dir, "upgrades")
cmd_target_point_df = s2.scan(ipc_dir, "cmd_target_point")
cmd_target_unit_df = s2.scan(ipc_dir, "cmd_target_unit")
# One row per player of every game with the tracker player_id of the events.
players_df = s2.players(ipc_dir)
details_df.collect_schema()

In [2]:
# DETAILS DATA FRAME
# This is a flattened version of the details structure https://github.com/sebosp/s2protocol-rs/blob/main/src/details.rs#L240
# The subfields player:Vec<> and the <lobby> are flattened here. Some fields are translated from enums to Strings.
details_df.select(pl.col("player_name").value_counts(sort=True)).collect()
# player_name: The player containing the clan
# The player region, 1,2,3 (enum for EU, US, KR, etc)
# player_toon_program_id: unsure
# realm: 1,2, unsure
# Race, internationalized, i.e. Protoss may be in english, korean, chinese.
# player_color_<r,g,b>: The chosen player color in the UI, RGB
# player_control: unsure, but used in the blizzard API for the toon/user
# player_team_id: configuration for 1v1 or 2v2, etc
# player_observe: 0 = None, 1 = spectator, 2 = Referee (TODO: needs validation, could be lobby_slot_observe, if same we can drop.)
# player_result: Undecided, Win, Loss, Tie
# player_working_set_slot_id: Position in the events, it references mostly this field on activity (camera position, etc)
# title: The title of the map.
# is_blizzard_map: bool
# time_utc: blizzard's epoch from some 2000-ish, translated as ext_datetime to human time.
# time_local_offset: timezone offset
# lobby_slot_user_id: Lobby position (up to 16 users in a game, some players, some observers)
# lobby_slot_observe: 0 = None, 1 = Spectator, 2 = Referee (TODO: needs validation, could be player_observe, if same we can drop.)
# lobby_slot_map_size_<x,y>
# cache_handle_region: "eu", "us", <region>.battle.net/<cache_handle_path>
# cache_handle_extension: "s2ma", etc.
# ext_fs_sha256: generated sha256 for uniqueness
# ext_fs_file_name: The path in which the replay is stored.
# ext_fs_id: A generated sequence from the snapshot generated process
# ext_datetime: The time in nanoseconds of the game translated to human date.

player_name
struct[2]
"{""VITZoun"",95}"
"{""LiquidCure"",94}"
"{""Classic"",91}"
"{""&lt;RBLN&gt;<sp/>ByuN"",81}"
"{""&lt;BASKKR&gt;<sp/>Reynor"",68}"
"{""&lt;BASKKR&gt;<sp/>trigger"",62}"
"{""&lt;PSISTM&gt;<sp/>MaxPax"",59}"
"{""VITMaru"",54}"
"{""FalconsSolar"",54}"


In [3]:
unit_born_df.collect_schema()

Schema([('unit_tag_index', UInt32),
        ('unit_tag_recycle', UInt32),
        ('unit_type_name', String),
        ('control_player_id', UInt8),
        ('upkeep_player_id', UInt8),
        ('x', Float32),
        ('y', Float32),
        ('creator_unit_tag_index', UInt32),
        ('creator_unit_tag_recycle', UInt32),
        ('creator_unit_type_name', String),
        ('creator_ability_name', String),
        ('ext_replay_loop', Int64),
        ('ext_replay_seconds', UInt32),
        ('ext_fs_id', UInt64),
        ('player_name', String)])

In [5]:
unit_born_df.select(pl.col("player_name").value_counts(sort=True)).collect()

player_name
struct[2]
"{""&lt;BASKKR&gt;<sp/>Reynor"",157044}"
"{null,112118}"
"{""Rogue"",98205}"
"{""FalconsSolar"",97930}"
"{""&lt;Weibo&gt;<sp/>SHIN"",83740}"
"{""&lt;BASKGG&gt;<sp/>Reynor"",80729}"
"{""WBGSHIN"",49526}"
"{""&lt;RBLN&gt;<sp/>ByuN"",37676}"
"{""LiquidCure"",35660}"


In [ ]:
details_df.filter(pl.col("ext_fs_id").eq(18780) & pl.col("lobby_slot_observe").eq(0)).select(
    pl.col("^lobby_slot_.*$"), "player_name", "player_working_set_slot_id"
).collect()

In [168]:
details_df.collect_schema()

Schema([('player_name', String),
        ('player_toon_region', UInt8),
        ('player_toon_program_id', UInt32),
        ('player_toon_realm', UInt32),
        ('player_toon_id', UInt64),
        ('player_race', String),
        ('player_color_a', UInt8),
        ('player_color_r', UInt8),
        ('player_color_g', UInt8),
        ('player_color_b', UInt8),
        ('player_control', UInt8),
        ('player_team_id', UInt8),
        ('player_handicap', UInt32),
        ('player_observe', UInt8),
        ('player_result', UInt8),
        ('player_working_set_slot_id', UInt8),
        ('player_hero', String),
        ('title', String),
        ('difficulty', String),
        ('is_blizzard_map', Boolean),
        ('time_utc', Int64),
        ('time_local_offset', Int64),
        ('restart_as_transition_map', Boolean),
        ('disable_recover_game', Boolean),
        ('description', String),
        ('image_file_path', String),
        ('campaign_index', UInt8),
        ('map_file_n

In [196]:
details_df.select(pl.col("player_name").value_counts(sort=True)).collect()
# player_name contains the clan
# player_race is Protoss, Terran, but also has hanzi (神族), korean? (테란) and strings like "Primal Zerg" or "Terraner" or ""
# player_team_id is either 0 or 1
# player_observe is always 0
# player_result can be 1, 2,0 and 3
#               1 and 2 are win,loss (not sure which is which). 
#               0 is probably undecided (or maybe teh status of observer?)
#               3 is probably tie (there are only 50 games with this status in this 12K replay dataset.
# player_working_set_slot_id null, 0..15
#                            not the same as lobby_slot_init_data_df working_set_slot_id
# player_hero always ""
# title is the name of the map
# is_blizzard_map seems filled
# time_utc is that weird "epoch" after 2000 or something
# time_local_offset is +- in millis (or nanos?)

player_name
struct[2]
"{""&lt;Mealen&gt;<sp/>Doombringer"",3225}"
"{""A.I. 2 (Elite)"",3060}"
"{""A.I. 3 (Elite)"",3004}"
"{""&lt;chezs&gt;<sp/>Sazed"",2406}"
"{""Sazed"",1880}"
"{""Cheater 2 (Insane)"",968}"
"{""Cheater 1 (Insane)"",818}"
"{""&lt;ƖIıIƖ&gt;<sp/>Sazed"",744}"
"{""Cheater 3 (Insane)"",189}"


In [166]:
details_df.filter([pl.col("ext_fs_id").eq(18780)]).collect()

player_name,player_toon_region,player_toon_program_id,player_toon_realm,player_toon_id,player_race,player_color_a,player_color_r,player_color_g,player_color_b,player_control,player_team_id,player_handicap,player_observe,player_result,player_working_set_slot_id,player_hero,title,difficulty,is_blizzard_map,time_utc,time_local_offset,restart_as_transition_map,disable_recover_game,description,image_file_path,campaign_index,map_file_name,mini_save,game_speed,default_difficulty,ext_fs_id,ext_datetime
str,u8,u32,u32,u64,str,u8,u8,u8,u8,u8,u8,u32,u8,u8,u8,str,str,str,bool,i64,i64,bool,bool,str,str,u8,str,bool,u8,u32,u64,datetime[ns]
"""TYTY""",3,21298,1,2341765,"""Terran""",255,180,20,30,2,1,100,0,2,2,"""""","""[ESL] Royal Blood""","""""",false,133343197785450190,324000000000,false,false,"""""","""""",0,"""""",false,4,3,18780,2023-07-20 00:42:58.545019
"""Dark""",3,21298,1,2333312,"""Zerg""",255,0,66,255,2,0,100,0,1,3,"""""","""[ESL] Royal Blood""","""""",false,133343197785450190,324000000000,false,false,"""""","""""",0,"""""",false,4,3,18780,2023-07-20 00:42:58.545019


In [197]:
upgrades_df.collect_schema()

Schema([('player_id', UInt8),
        ('name', String),
        ('count', Int32),
        ('ext_replay_loop', Int64),
        ('ext_replay_seconds', UInt32),
        ('ext_fs_id', UInt64)])

In [202]:
upgrades_df.select(pl.col("name").value_counts(sort=True)).collect()
# player_id is 0..15, 0 is only 20 occurrences, it seems starts at 1, not sure if it's one of the working_set_slot_ids
# name: the name of the upgrade 

name
struct[2]
"{""SprayZerg"",182513}"
"{""SprayProtoss"",83708}"
"{""SprayTerran"",78628}"
"{""RewardDanceOverlord"",15137}"
"{""RewardDanceRoach"",14789}"
"{""RewardDanceInfestor"",14196}"
"{""zerglingmovementspeed"",13069}"
"{""RewardDanceMule"",11240}"
"{""RewardDanceStalker"",10828}"


In [ ]:
# The players of the events (stats, upgrades, unit_born, unit_died) are identified by the tracker
# player_id, s2.players() maps them to the details by the player name of unit_born. The
# player_working_set_slot_id + 1 is not always the player_id, i.e. with observers.
players_df.collect_schema()

latest_replay = details_df.sort("ext_datetime", descending=True).limit(15).collect()
latest_replay.select("ext_fs_id", "ext_datetime", "title", "player_name", "player_result").head()

In [ ]:
latest_replay_id = latest_replay["ext_fs_id"][0]

top_player = (
    players_df.group_by("player")
    .agg(pl.len().alias("games"), (pl.col("player_result") == "Win").mean().alias("win_rate"))
    .sort("games", descending=True)
    .collect()
)
top_player.head()

In [ ]:
unit_born_latest_replay = (
    (
        unit_born_df.filter(pl.col("ext_fs_id") == latest_replay_id)
        .with_columns(
            pl.concat_str(pl.col("unit_type_name"), pl.col("unit_tag_index")).alias(
                "name_index"
            ),
            pl.concat_str(
                [
                    pl.col("creator_unit_type_name").fill_null("sys"),
                    pl.col("creator_ability_name").fill_null("sys"),
                    pl.col("unit_type_name"),
                ],
                separator=".",
            ).alias("ability_creates_unit_type"),
        )
        .select(
            [
                pl.col("x"),
                pl.col("y"),
                pl.col("name_index"),
                pl.col("unit_tag_index"),
                pl.col("unit_type_name"),
                pl.col("ext_replay_loop"),
                pl.col("ability_creates_unit_type"),
            ]
        )
        .sort("ext_replay_loop")
    )
    .collect()
)

unit_born_latest_replay_fig = px.scatter(
    unit_born_latest_replay,
    x="x",
    y="y",
    color="ability_creates_unit_type",
    symbol="ability_creates_unit_type",
    width=1200,
    height=800,
    range_x=(0, 200),
    range_y=(0, 200),
)
unit_born_latest_replay_fig.show()
unit_born_latest_replay_fig.write_html("static/latest_map_unit_born.html")
unit_born_latest_replay.head()

In [ ]:
unit_died_latest_replay = (
    unit_died_df.filter(pl.col("ext_fs_id") == latest_replay_id)
    .select([pl.col("unit_died_name"), pl.col("x"), pl.col("y")])
    .collect()
)
unit_died_latest_replay_fig = px.scatter(
    unit_died_latest_replay,
    x="x",
    y="y",
    color="unit_died_name",
    width=1200,
    height=800,
    range_x=(0, 200),
    range_y=(0, 200),
)
unit_died_latest_replay_fig.show()
unit_died_latest_replay_fig.write_html("static/latest_map_unit_died.html")

In [ ]:
# The rows of the replay are filtered before the join, so that only them are joined.
unit_died_latest_map = (
    unit_died_df.filter(
        (pl.col("ext_fs_id") == latest_replay_id)
        # Units auto attack "invisible targets" and when they hit such target
        # It is deregistered and a UnitDiedEvent is emitted.
        # We should rename this to "Deregister".
        & (pl.col("unit_died_name") != "InvisibleTargetDummy")
        # Units may "die" sometimes with no killer name, for example:
        # - Once Larva is used.
        # - When Broodlings, BroodlingsEscorts, Changelings expire
        # Units emit UnitDiedEvents when they cast an ability that expires:
        # - A Sentry casts a ForceField, this will expire and emit such event.
        # - An Adept casts a AdeptPhaseShift
        # - A Viper casts Parasitic Bomb
        & (pl.col("unit_killer_name") != "")
    )
    .join(
        players_df.select("ext_fs_id", pl.col("player_id").alias("killer_player_id"), "player"),
        on=["ext_fs_id", "killer_player_id"],
        how="left",
    )
    .select(
        [
            pl.col("player").alias("killer_player"),
            pl.col("unit_died_name"),
            pl.col("unit_tag_index"),
            pl.col("unit_killer_name"),
            pl.col("killer_unit_tag_index"),
            pl.col("ext_replay_loop"),
            s2.game_seconds(),
        ]
    )
)
unit_died_latest_map.collect().head()

In [ ]:
# The three frequencies share the scan and the filters of unit_died_latest_map, collect_all runs
# them as one plan.
unit_killer_to_died_freq, unit_killer_freq, unit_died_freq = pl.collect_all(
    [
        unit_died_latest_map.group_by(group).agg(pl.len().alias("count")).sort("count")
        for group in [["unit_killer_name", "unit_died_name"], ["unit_killer_name"], ["unit_died_name"]]
    ]
)
unit_killer_to_died_freq_hist = px.histogram(
    unit_killer_to_died_freq, x="unit_killer_name", y="count", color="unit_died_name"
).update_xaxes(categoryorder="total ascending")
# Q: What is the unit that kills other units the most across all my game history?
# A: LurkerMPBurrowed killed 21k marines:

unit_killer_to_died_freq_hist.show()
unit_killer_to_died_freq_hist.write_html(
    "static/latest_map_unit_killer_to_died_freq.html"
)
# unit_killer_pie = px.pie(unit_killer_freq, names="unit_killer_name", values="count")
# unit_killer_pie.show()

In [ ]:
# Units like lurkers or broodlords attack "InvisibleTargetDummies"
unit_attack_target = (
    unit_died_df.filter(
        (pl.col("ext_fs_id") == latest_replay_id)
        & (pl.col("unit_died_name") == "InvisibleTargetDummy")
    )
    .join(
        players_df.select("ext_fs_id", pl.col("player_id").alias("killer_player_id"), "player"),
        on=["ext_fs_id", "killer_player_id"],
        how="left",
    )
    .select(
        [
            pl.col("player").alias("killer_player"),
            pl.col("unit_tag_index"),
            pl.col("unit_killer_name"),
            pl.col("killer_unit_tag_index"),
            pl.col("ext_replay_loop"),
            s2.game_seconds(),
        ]
    )
)

In [ ]:
# The units that attacked the invisible target dummies the most.
unit_attack_freq = (
    unit_attack_target.group_by("unit_killer_name")
    .agg(pl.len().alias("count"))
    .sort("count")
    .collect()
)
unit_attack_freq

In [ ]:
unit_born_drones_latest_map = (
    unit_born_latest_replay.lazy()
    .filter(pl.col("unit_type_name") == "Drone")
    .group_by(["x", "y", "unit_type_name"])
    .agg(pl.len().alias("count"))
    .collect()
)
fig = px.scatter(
    unit_born_drones_latest_map,
    x="x",
    y="y",
    color="unit_type_name",
    size="count",
    render_mode="svg",
    width=1600,
    height=800,
)
fig.show()
fig.write_html("static/latest_map_drone_born_scatter.html")

## Time series with functime

The functime examples (forecasting the race balance, upgrade timing trends, a global model of the workers per minute and the features of the army curves) are in [functime.ipynb](./functime.ipynb).